In [20]:
# import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

In [21]:
# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

In [22]:
# LOAD DATA
print("\n LOADING DATA...")
print("-" * 80)

# Load the dataset
df = pd.read_csv('../data/raw/BD_growth_prog_anon.csv')

print(f"✓ Data loaded successfully")
print(f"  Total rows: {len(df):,}")
print(f"  Total columns: {len(df.columns)}")


 LOADING DATA...
--------------------------------------------------------------------------------
✓ Data loaded successfully
  Total rows: 486,267
  Total columns: 29


In [23]:
# BASIC DATASET INFORMATION
print("\n DATASET OVERVIEW")
print("-" * 80)

print("\nColumn Names and Types:")
print(df.dtypes)

print("\nFirst 5 rows:")
print(df.head())

print("\nDataset Shape:", df.shape)
print(f"Memory Usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")


 DATASET OVERVIEW
--------------------------------------------------------------------------------

Column Names and Types:
Unnamed: 0                   int64
district                    object
upazila                      int64
union                        int64
village                      int64
hh_id                        int64
child_id                     int64
date                        object
gender                      object
dob                         object
height                     float64
weight                     float64
cbmi                       float64
zlen                       float64
flen                       float64
zwei                       float64
fwei                       float64
zwfl                       float64
fwfl                       float64
zbmi                       float64
fbmi                       float64
flag_duplicated_day          int64
flag_duplicated_quarter      int64
flag_obs_number              int64
flag_different_dob           int64


In [25]:
# MISSING VALUES ANALYSIS
print("\n MISSING VALUES ANALYSIS")
print("-" * 80)

missing_data = pd.DataFrame({
    'Column': df.columns,
    'Missing_Count': df.isnull().sum(),
    'Missing_Percentage': (df.isnull().sum() / len(df) * 100).round(2)
})
missing_data = missing_data[missing_data['Missing_Count'] > 0].sort_values(
    'Missing_Percentage', ascending=False
)

if len(missing_data) > 0:
    print("\nColumns with Missing Values:")
    print(missing_data.to_string(index=False))
else:
    print("✓ No missing values found")


 MISSING VALUES ANALYSIS
--------------------------------------------------------------------------------

Columns with Missing Values:
         Column  Missing_Count  Missing_Percentage
           zwfl          21123                4.34
           fwfl          21123                4.34
           zlen          11186                2.30
           flen          11186                2.30
           zwei          11186                2.30
           fwei          11186                2.30
           zbmi          11186                2.30
           fbmi          11186                2.30
           cbmi           3884                0.80
         height           3064                0.63
         weight           3064                0.63
            dob            795                0.16
flag_dob_number            795                0.16


In [26]:
# UNIQUE CHILDREN AND MEASUREMENTS
print("\n CHILDREN AND MEASUREMENTS STATISTICS")
print("-" * 80)

n_unique_children = df['child_id'].nunique()
n_total_measurements = len(df)
avg_measurements_per_child = n_total_measurements / n_unique_children

print(f"Unique Children: {n_unique_children:,}")
print(f"Total Measurements: {n_total_measurements:,}")
print(f"Average Measurements per Child: {avg_measurements_per_child:.2f}")

# Distribution of measurements per child
measurements_per_child = df.groupby('child_id').size()
print(f"\nMeasurements Distribution:")
print(f"  Min: {measurements_per_child.min()}")
print(f"  Max: {measurements_per_child.max()}")
print(f"  Median: {measurements_per_child.median():.0f}")
print(f"  Mean: {measurements_per_child.mean():.2f}")

# Children with at least 2 measurements (needed for prediction)
children_with_2plus = (measurements_per_child >= 2).sum()
print(f"\nChildren with 2+ measurements: {children_with_2plus:,} ({children_with_2plus/n_unique_children*100:.1f}%)")


 CHILDREN AND MEASUREMENTS STATISTICS
--------------------------------------------------------------------------------
Unique Children: 66,712
Total Measurements: 486,267
Average Measurements per Child: 7.29

Measurements Distribution:
  Min: 1
  Max: 39
  Median: 7
  Mean: 7.29

Children with 2+ measurements: 60,079 (90.1%)


In [27]:
# GENDER ANALYSIS
print("\n GENDER ANALYSIS")
print("-" * 80)

print("\nGender Distribution:")
gender_dist = df['gender'].value_counts()
print(gender_dist)
print(f"  Male: {gender_dist.get('M', 0) / len(df) * 100:.1f}%")
print(f"  Female: {gender_dist.get('F', 0) / len(df) * 100:.1f}%")


 GENDER ANALYSIS
--------------------------------------------------------------------------------

Gender Distribution:
gender
M    246373
F    239894
Name: count, dtype: int64
  Male: 50.7%
  Female: 49.3%


In [28]:
# MEASUREMENT VARIABLES ANALYSIS
print("\n MEASUREMENT VARIABLES ANALYSIS")
print("-" * 80)

measurement_vars = ['height', 'weight', 'cbmi']
print("\nBasic Statistics for Measurements:")
print(df[measurement_vars].describe())

# Check for outliers/impossible values
print("\nPotential Issues:")
print(f"  Heights <= 0: {(df['height'] <= 0).sum()}")
print(f"  Weights <= 0: {(df['weight'] <= 0).sum()}")
print(f"  Heights > 200cm: {(df['height'] > 200).sum()}")
print(f"  Weights > 100kg: {(df['weight'] > 100).sum()}")


 MEASUREMENT VARIABLES ANALYSIS
--------------------------------------------------------------------------------

Basic Statistics for Measurements:
              height         weight      cbmi
count  483203.000000  483203.000000  482383.0
mean       84.067974      11.823615       inf
std       148.992975      47.308176       NaN
min         0.000000       0.000000       0.0
25%        74.000000       9.000000      14.4
50%        84.000000      11.000000      15.7
75%        94.000000      14.000000      17.8
max     97100.000000   14799.000000       inf

Potential Issues:
  Heights <= 0: 37
  Weights <= 0: 31
  Heights > 200cm: 151
  Weights > 100kg: 318


In [29]:
# Z-SCORE ANALYSIS
print("\n Z-SCORE ANALYSIS")
print("-" * 80)

z_score_vars = ['zlen', 'zwei', 'zwfl', 'zbmi']
print("\nZ-Score Statistics:")
print(df[z_score_vars].describe())

# WHO flag analysis
flag_vars = ['flen', 'fwei', 'fwfl', 'fbmi']
print("\nWHO Flags (measurements outside normal bounds):")
for flag in flag_vars:
    flagged = df[flag].sum()
    print(f"  {flag}: {flagged:,} ({flagged/len(df)*100:.2f}%)")


 Z-SCORE ANALYSIS
--------------------------------------------------------------------------------

Z-Score Statistics:
                zlen           zwei           zwfl      zbmi
count  475081.000000  475081.000000  465144.000000  475081.0
mean       -1.982482      -0.959878       0.053051       inf
std        42.713689      26.275391       2.176199       NaN
min       -31.800000     -11.200000     -14.600000     -16.9
25%        -3.500000      -1.900000      -1.000000      -0.9
50%        -1.700000      -1.000000      -0.100000       0.0
75%        -0.600000      -0.300000       1.000000       1.4
max     27587.500000   10648.300000     142.100000       inf

WHO Flags (measurements outside normal bounds):
  flen: 45,103.0 (9.28%)
  fwei: 5,183.0 (1.07%)
  fwfl: 13,165.0 (2.71%)
  fbmi: 24,083.0 (4.95%)


In [30]:
# DATA QUALITY FLAGS ANALYSIS
print("\n DATA QUALITY FLAGS ANALYSIS")
print("-" * 80)

quality_flags = [
    'flag_duplicated_day',
    'flag_duplicated_quarter',
    'flag_different_dob',
    'flag_under_zero',
    'flag_no_match',
    'flag_flip'
]

print("\nData Quality Issues:")
for flag in quality_flags:
    if flag in df.columns:
        flagged = df[flag].sum()
        print(f"  {flag}: {flagged:,} ({flagged/len(df)*100:.2f}%)")


 DATA QUALITY FLAGS ANALYSIS
--------------------------------------------------------------------------------

Data Quality Issues:
  flag_duplicated_day: 13,793 (2.84%)
  flag_duplicated_quarter: 269,271 (55.38%)
  flag_different_dob: 754 (0.16%)
  flag_under_zero: 6 (0.00%)
  flag_no_match: 795 (0.16%)
  flag_flip: 1,473 (0.30%)


In [31]:
# TEMPORAL ANALYSIS
print("\n TEMPORAL ANALYSIS")
print("-" * 80)

# Convert dates
df['date'] = pd.to_datetime(df['date'])
df['dob'] = pd.to_datetime(df['dob'])

print(f"\nMeasurement Date Range:")
print(f"  First: {df['date'].min()}")
print(f"  Last: {df['date'].max()}")
print(f"  Span: {(df['date'].max() - df['date'].min()).days} days")

print(f"\nBirth Date Range:")
print(f"  Earliest: {df['dob'].min()}")
print(f"  Latest: {df['dob'].max()}")

# Calculate age at measurement
df['age_days'] = (df['date'] - df['dob']).dt.days
df['age_months'] = df['age_days'] / 30.44

print(f"\nAge at Measurement:")
print(f"  Min: {df['age_months'].min():.1f} months")
print(f"  Max: {df['age_months'].max():.1f} months")
print(f"  Mean: {df['age_months'].mean():.1f} months")

# Children measured at negative age (data error)
negative_age = (df['age_days'] < 0).sum()
if negative_age > 0:
    print(f"\n⚠ WARNING: {negative_age} measurements with negative age (data quality issue)")


 TEMPORAL ANALYSIS
--------------------------------------------------------------------------------

Measurement Date Range:
  First: 2018-04-01 00:00:00
  Last: 2021-10-01 00:00:00
  Span: 1279 days

Birth Date Range:
  Earliest: 2013-10-01 00:00:00
  Latest: 2021-07-01 00:00:00

Age at Measurement:
  Min: -6.0 months
  Max: 96.0 months
  Mean: 32.0 months

⚠ WARNING: 5 measurements with negative age (data quality issue)


In [32]:
# KEY FINDINGS SUMMARY
print("\n" + "=" * 80)
print("KEY FINDINGS & RECOMMENDATIONS")
print("=" * 80)

print("\n✓ STRENGTHS:")
print("  1. Large dataset with 480k+ measurements")
print(f"  2. {children_with_2plus:,} children have 2+ measurements (good for training)")
print("  3. Both height and weight recorded with WHO z-scores")
print("  4. Temporal data spanning multiple years")

print("\n⚠ ISSUES TO ADDRESS:")
issues = []

if missing_data.shape[0] > 0:
    issues.append(f"  1. Missing values in {len(missing_data)} columns")
    
if (df['height'] <= 0).sum() + (df['weight'] <= 0).sum() > 0:
    issues.append("  2. Invalid measurements (height/weight <= 0)")
    
if df['flag_no_match'].sum() > 0:
    issues.append(f"  3. {df['flag_no_match'].sum():,} records without birth date info")
    
if df['flag_duplicated_quarter'].sum() > 0:
    issues.append(f"  4. {df['flag_duplicated_quarter'].sum():,} duplicate measurements in same quarter")
    
if df['flag_flip'].sum() > 0:
    issues.append(f"  5. {df['flag_flip'].sum():,} records with height/weight potentially flipped")

for issue in issues:
    print(issue)


KEY FINDINGS & RECOMMENDATIONS

✓ STRENGTHS:
  1. Large dataset with 480k+ measurements
  2. 60,079 children have 2+ measurements (good for training)
  3. Both height and weight recorded with WHO z-scores
  4. Temporal data spanning multiple years

⚠ ISSUES TO ADDRESS:
  1. Missing values in 13 columns
  2. Invalid measurements (height/weight <= 0)
  3. 795 records without birth date info
  4. 269,271 duplicate measurements in same quarter
  5. 1,473 records with height/weight potentially flipped
